<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [2]:
#import piplite
#await piplite.install(['folium'])
#await piplite.install(['pandas'])

%pip install folium pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import folium
import pandas as pd

In [4]:
pd

<module 'pandas' from 'c:\\Users\\DITEC AUTO\\AppData\\Local\\Programs\\Python\\Python310\\lib\\site-packages\\pandas\\__init__.py'>

In [5]:
# Import folium MarkerCluster plugin
from folium.plugins import MarkerCluster
# Import folium MousePosition plugin
from folium.plugins import MousePosition
# Import folium DivIcon plugin
from folium.features import DivIcon

If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


In [6]:
## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


In [7]:
# Download and read the `spacex_launch_geo.csv`
#from js import fetch
#import io

#URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
#resp = await fetch(URL)
#spacex_csv_file = io.BytesIO((await resp.arrayBuffer()).to_py())
#spacex_df=pd.read_csv(spacex_csv_file)

URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"

spacex_df = pd.read_csv(URL)

spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


Now, you can take a look at what are the coordinates for each site.


In [8]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [9]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


In [10]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [11]:
# Initial the map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Create a circle and a text label for each launch site
for index, row in launch_sites_df.iterrows():

    coordinate = [row['Lat'], row['Long']]

    # Add circle
    circle = folium.Circle(
        coordinate,
        radius=1000,
        color='#d35400',
        fill=True
    ).add_child(
        folium.Popup(row['Launch Site'])
    )

    # Add text marker
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site']
        )
    )

    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


Answer:
1. Are all launch sites in proximity to the Equator line?

Answer: Yes, relatively.

Explanation:
All four launch sites are located in the southern part of the United States, at relatively low latitudes compared to many other locations around the world. Although they are not close to the Equator itself, they are much closer than launch sites located in northern regions. Launching from lower latitudes provides an advantage because rockets can benefit from the Earth's rotational speed, reducing the energy required to reach orbit.

2. Are all launch sites in very close proximity to the coast?

Answer: Yes.

Explanation:
All four launch sites are located very close to the coastline. This is intentional because rockets are typically launched over the ocean, minimizing the risk to populated areas in the event of debris or a launch failure. Coastal launch sites also provide safe flight paths for missions heading eastward over the Atlantic or southward over the Pacific.

In [12]:
# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


In [13]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


In [14]:
marker_cluster = MarkerCluster()


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


In [15]:
# Create a new marker_color column from the landing outcome.
# class = 1 -> green (successful landing)
# class = 0 -> red   (failed landing)

spacex_df['marker_color'] = spacex_df['class'].apply(
    lambda x: 'green' if x == 1 else 'red'
)

spacex_df[['Launch Site', 'class', 'marker_color']].head(10)


,Launch Site,class,marker_color
0,CCAFS LC-40,0,red
1,CCAFS LC-40,0,red
2,CCAFS LC-40,0,red
3,CCAFS LC-40,0,red
4,CCAFS LC-40,0,red
5,CCAFS LC-40,0,red
6,CCAFS LC-40,0,red
7,CCAFS LC-40,0,red
8,CCAFS LC-40,0,red
9,CCAFS LC-40,0,red


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [16]:
# Add the MarkerCluster to the current site_map
site_map.add_child(marker_cluster)

# Add one marker for every launch record
for index, record in spacex_df.iterrows():
    folium.Marker(
        location=[record['Lat'], record['Long']],
        popup=f"{record['Launch Site']} | class={record['class']}",
        icon=folium.Icon(
            color=record['marker_color'],
            icon='info-sign'
        )
    ).add_to(marker_cluster)

site_map

#this is used because VS Code is not trusting the notebook
site_map.save("site_map.html")

In [17]:

site_map
site_map.save("site_map.html")
site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


In [18]:
# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [19]:
# Add MousePosition so the map shows latitude/longitude under the cursor
formatter = "function(num) {return L.Util.formatNum(num, 5);};"

mouse_position = MousePosition(
    position='topleft',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)

site_map.save("site_map.html")

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [20]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # Great-circle distance in kilometers
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

# Use CCAFS SLC-40 for the proximity example
launch_site_row = launch_sites_df[
    launch_sites_df['Launch Site'] == 'CCAFS SLC-40'
].iloc[0]

launch_site_lat = float(launch_site_row['Lat'])
launch_site_lon = float(launch_site_row['Long'])
launch_site_coordinates = [launch_site_lat, launch_site_lon]

print("Launch site coordinates:", launch_site_coordinates)


Launch site coordinates: [28.56319718, -80.57682003]


In [21]:
# Closest coastline point selected with MousePosition
coastline_lat = 28.56260
coastline_lon = -80.56802
coastline_coordinates = [coastline_lat, coastline_lon]

distance_coastline = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    coastline_lat,
    coastline_lon
)

print(f"Distance from CCAFS SLC-40 to coastline: {distance_coastline:.2f} km")


Distance from CCAFS SLC-40 to coastline: 0.86 km


*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [22]:
print("Launch site:", launch_site_coordinates)
print("Coastline point:", coastline_coordinates)
print(f"Distance: {distance_coastline:.2f} km")


Launch site: [28.56319718, -80.57682003]
Coastline point: [28.5626, -80.56802]
Distance: 0.86 km


In [27]:
# Add a marker at the selected coastline point and show its distance
distance_marker = folium.Marker(
    location=coastline_coordinates,
    icon=DivIcon(
        icon_size=(120, 30),
        icon_anchor=(0, 0),
        html=(
            '<div style="font-size: 12px; color:#d35400;">'
            f'<b>{distance_coastline:.2f} KM</b>'
            '</div>'
        )
    )
)

site_map.add_child(distance_marker)

site_map.save("site_map.html")


site_map


*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [29]:
# Draw a line from the launch site to the selected coastline point
distance_line = folium.PolyLine(
    locations=[launch_site_coordinates, coastline_coordinates],
    weight=2
)

site_map.add_child(distance_line)
site_map


Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line between a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [31]:
# Helper for additional proximity points such as railway, highway, and city.
# Use MousePosition to read coordinates from the map, then call this function.

def add_proximity_point(name, lat, lon):
    destination = [lat, lon]

    distance_km = calculate_distance(
        launch_site_lat,
        launch_site_lon,
        lat,
        lon
    )

    label = folium.Marker(
        location=destination,
        icon=DivIcon(
            icon_size=(180, 30),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size: 12px; color:#d35400;">'
                f'<b>{name}: {distance_km:.2f} KM</b>'
                '</div>'
            )
        )
    )

    line = folium.PolyLine(
        locations=[launch_site_coordinates, destination],
        weight=2
    )

    site_map.add_child(label)
    site_map.add_child(line)

    return distance_km

# Example usage after reading coordinates with MousePosition:
# add_proximity_point("Railway", railway_lat, railway_lon)
# add_proximity_point("Highway", highway_lat, highway_lon)
# add_proximity_point("City", city_lat, city_lon)

# Coordinates selected with MousePosition
railway_lat = 28.56226
railway_lon = -80.57741

highway_lat = 28.56232
highway_lon = -80.57068

city_lat = 28.39998
city_lon = -80.60459

# Add proximity points, distance labels, and lines
distance_railway = add_proximity_point(
    "Railway", railway_lat, railway_lon
)

distance_highway = add_proximity_point(
    "Highway", highway_lat, highway_lon
)

distance_city = add_proximity_point(
    "City", city_lat, city_lon
)

print(f"Railway distance: {distance_railway:.2f} km")
print(f"Highway distance: {distance_highway:.2f} km")
print(f"City distance: {distance_city:.2f} km")

# Zoom map to include launch site and all proximity points
site_map.fit_bounds([
    launch_site_coordinates,
    coastline_coordinates,
    [railway_lat, railway_lon],
    [highway_lat, highway_lon],
    [city_lat, city_lon]
])

site_map



Railway distance: 0.12 km
Highway distance: 0.61 km
City distance: 18.36 km


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


### Findings

For the selected launch site, CCAFS SLC-40:

- **Railway proximity:** Yes. The nearest railway point is approximately **0.12 km** from the launch site.
- **Highway proximity:** Yes. The nearest major road/highway is approximately **0.61 km** away.
- **Coastline proximity:** Yes. The Atlantic coastline is approximately **0.86 km** from the launch site.
- **Distance from cities:** The selected nearby populated area is much farther away, at approximately **18.36 km**.

These results suggest that the launch site is located close to transportation infrastructure and the coastline, while maintaining a much greater distance from populated areas. This location pattern is suitable for launch operations because it provides logistical access while reducing risks to nearby population centers.

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
